# Session 12 · Exercises

*Course material, licence CC BY 4.0.*

Work in pairs. The lecturer chooses which exercises to set; there are more than fit into 30 minutes. *Paper* and *Predict* tasks are answered in the markdown cell below the task, before any code is run; *Code* tasks in the empty code cells. Exercises marked *Stretch* are for pairs who finish early.

## Setup

Run these cells once before the exercises that use data.

In [ ]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

In [ ]:
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")     # statsmodels convergence notes

# The monthly number of Airbnb reviews in Berlin, as in the case study (January 2016 to May 2026)
reviews = pd.read_parquet("case-study/data/airbnb/reviews_monthly.parquet")
y = reviews.groupby("month")["n_reviews"].sum()["2016":"2026-05"].rename("reviews")
y.index.freq = "MS"
train, test = y[:"2025-05"], y["2025-06":]                      # hold-out year June 2025 to May 2026
covid = pd.Series((y.index >= "2020-03-01") & (y.index <= "2021-12-01"), index=y.index)


def mae(actual, forecast):
    return float(np.mean(np.abs(np.asarray(actual) - np.asarray(forecast))))


print(len(y), "months;", len(train), "for training,", len(test), "for the hold-out year")

## Part 1 · Time series, baselines and exponential smoothing

### 1.1 · Which chart is real?

*Class, 5 min*

> The lecturer shows four monthly series: one real series and three random walks. Vote: which one is real, and why do you think so?

*Source:* [Wickham et al. (2010), "Graphical inference for infovis", IEEE TVCG](https://doi.org/10.1109/TVCG.2010.161)

Your answer:

### 1.2 · Class forecast tournament

*Class, 15 min*

> The cell below prints 36 months of Berlin reviews, June 2022 to May 2025. Without a computer model, forecast the next three months (June, July and August 2025) by hand and write your three numbers down. The lecturer collects the forecasts and compares them with the actual values and with the seasonal naive forecast.

*Source:* original exercise for this course; background: [Hyndman & Athanasopoulos, *Forecasting: Principles and Practice* (3rd ed.), simple methods](https://otexts.com/fpp3/toolbox.html#sec-simple-methods)

In [ ]:
y["2022-06":"2025-05"].to_frame().T

Your answer:

### 1.3 · Spurious correlation

*Code, 10 min*

> Generate two independent random walks of length 100 (`np.cumsum` of standard normal noise) and compute their correlation. Repeat 100 times. How often is |r| above 0.5? What happens if you correlate the differences (`np.diff`) instead?

*Source:* [Granger & Newbold (1974), "Spurious regressions in econometrics", *Journal of Econometrics*](https://doi.org/10.1016/0304-4076(74)90034-7)

### 1.4 · Exponential smoothing by hand

*Paper, 10 min*

> Simple exponential smoothing with α = 0.5 on the series 10, 14, 12, 16, starting the level at the first value. Compute the level after each value and the forecast for the next period.

*Source:* [*Forecasting: Principles and Practice*, simple exponential smoothing](https://otexts.com/fpp3/exponential-smoothing.html#sec-ses)

Your answer:

### 1.5 · January against February

*Discussion, 5 min*

> February has fewer reviews than March. Is that lower demand or a shorter month? How would you check it with `y`?

*Source:* [*Forecasting: Principles and Practice*, calendar adjustments](https://otexts.com/fpp3/decomposition.html#sec-transformations)

Your answer:

### 1.6 · How much history for ETS?

*Code, 15 min*

> The lecture fits ETS (additive error, damped additive trend, additive seasonality, on the log counts) on `train` from January 2022. Fit the same model from July 2021 and from January 2023:
>
> ```python
> from statsmodels.tsa.exponential_smoothing.ets import ETSModel
> model = ETSModel(np.log(train[start:]), error="add", trend="add", damped_trend=True,
>                  seasonal="add", seasonal_periods=12).fit(disp=False)
> ```
>
> For each start, compute the MAE on `test`, the width of the 95 % prediction interval in the first and the last month (as a share of the forecast; `get_prediction(...).summary_frame(alpha=0.05)`, back-transformed with `np.exp`) and the number of months inside the interval. What does this tell you about the reliability of a model fitted on a short history?

*Source:* original exercise for this course, from the exercises of the case study; background: [*Forecasting: Principles and Practice*, ETS models](https://otexts.com/fpp3/ets.html)

### 1.7 · A lockdown dummy

*Code, 15 min, Stretch*

> The airline model (`SARIMAX(..., order=(0, 1, 1), seasonal_order=(0, 1, 1, 12))` on the log counts) can treat the pandemic in three ways: mark the months as missing (`np.log(train).where(~covid[train.index])`), ignore the break, or add a dummy as an exogenous variable (`exog=covid[train.index].astype(float)`; forecasting then needs the future values of the dummy: zeros). Fit all three, compare the MAE on `test` and the width of the interval after twelve months. What does the dummy coefficient say?

*Source:* original exercise for this course, from the exercises of the case study; background: [statsmodels: SARIMAX](https://www.statsmodels.org/stable/statespace.html)

## Part 2 · Machine learning forecasts and backtesting

### 2.1 · Shuffled k-fold on a time series

*Predict, 5 min*

> A colleague evaluates a lag-feature model of the monthly reviews with `KFold(5, shuffle=True)`. Predict: is the score too optimistic, too pessimistic or fine? Why?

*Source:* [*Forecasting: Principles and Practice*, time series cross-validation](https://otexts.com/fpp3/toolbox.html#sec-tscv)

Your answer:

### 2.2 · MASE of 1.2

*Paper, 3 min*

> A forecast has a MASE of 1.2. What does it mean?

*Source:* [Hyndman & Koehler (2006), "Another look at measures of forecast accuracy", *International Journal of Forecasting*](https://doi.org/10.1016/j.ijforecast.2006.03.001)

Your answer:

### 2.3 · MASE by hand

*Paper, 5 min*

> A quarterly series (season length m = 4) has the training values 10, 14, 8, 12, 11, 15, 9, 13. The next two quarters are 12 and 16; a model forecast 13 and 14. Compute the MAE of the model, the scale of MASE (the mean absolute error of the seasonal naive forecast on the training data) and the MASE. Is the model better than the seasonal naive rule?

*Source:* original exercise for this course; background: [Hyndman & Koehler (2006)](https://doi.org/10.1016/j.ijforecast.2006.03.001)

Your answer:

### 2.4 · Bookable nights are not demand

*Code, 15 min*

> Load the calendar of the snapshot (`pd.read_parquet("case-study/data/airbnb/calendar.parquet")`: one row per listing and night, with `available` True or False) and compute the share of nights still available per month for the next twelve months. Plot it. Why is this not a forecast of demand? Name two reasons why a night can be unavailable.

*Source:* original exercise for this course, from the exercises of the case study; background: [Inside Airbnb: Data assumptions](https://insideairbnb.com/data-assumptions/)

### 2.5 · Backtest origins every month

*Code, pairs, 20 min*

> The lecture backtests from six origins, three months apart (March 2024 to June 2025), each with a 12-month horizon. Repeat the backtest for the seasonal naive forecast and the seasonal naive forecast times growth with origins every month over the same period (16 origins). Report the mean and the worst MAE per model. Do the rankings change? Why are neighbouring folds now strongly correlated, and what does that mean for the number of "independent" tests?

*Source:* original exercise for this course, from the exercises of the case study; background: [*Forecasting: Principles and Practice*, time series cross-validation](https://otexts.com/fpp3/toolbox.html#sec-tscv)

### 2.6 · Level or growth as the target?

*Code, 20 min, Stretch*

> The lag model of the lecture predicts the yearly growth `log y - log y.shift(12)` from `lag12 = log y.shift(12)`, `lag24` and the month, and drops the rows touched by the pandemic (`covid`, shifted by 12 and 24 months). Build the same table, but with the level `log y` as the target. Fit `HistGradientBoostingRegressor(min_samples_leaf=5, random_state=0)` on the rows up to May 2025 and forecast the hold-out year with both targets. Compare the MAE and plot both forecasts against `test`. Which target can forecast values above anything seen in training, and why?

*Source:* original exercise for this course; background: [scikit-learn example: Lagged features for time series forecasting](https://scikit-learn.org/stable/auto_examples/applications/plot_time_series_lagged_features.html)